# RadiologyBLIP: Fine-Tuning BLIP for Medical Visual Question Answering


In [ ]:
from pathlib import Path
import json
import os
import random
import re
import unicodedata
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import GroupShuffleSplit
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BlipProcessor, BlipForQuestionAnswering

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

IMAGE_DIR =  "images"
DATA_JSON = "vqa_rad.json"
MODEL_NAME = "Salesforce/blip-vqa-base"

MAX_TRAIN_SAMPLES = 1200
MAX_TEST_SAMPLES = 300
EPOCHS = 1
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 2e-5

OUTPUT_DIR = "radiologyblip_output"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

c:\Users\spars\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [2]:
with open(DATA_JSON, "r", encoding="utf-8") as f:
    raw_records = json.load(f)

print("Number of raw records:", len(raw_records))

print("\nFirst record:")
print(raw_records[0])

print("\nAvailable fields:")
print(raw_records[0].keys())

Number of raw records: 2248

First record:
{'qid': '0', 'phrase_type': 'freeform', 'qid_linked_id': '03f451ca-de62-4617-9679-e836026a7642', 'image_case_url': 'https://medpix.nlm.nih.gov/case?id=48e1dd0e-8552-46ad-a354-5eb55be86de6', 'image_name': 'synpic54610.jpg', 'image_organ': 'HEAD', 'evaluation': 'not evaluated', 'question': 'Are regions of the brain infarcted?', 'question_rephrase': 'NULL', 'question_relation': 'NULL', 'question_frame': 'NULL', 'question_type': 'PRES', 'answer': 'Yes', 'answer_type': 'CLOSED'}

Available fields:
dict_keys(['qid', 'phrase_type', 'qid_linked_id', 'image_case_url', 'image_name', 'image_organ', 'evaluation', 'question', 'question_rephrase', 'question_relation', 'question_frame', 'question_type', 'answer', 'answer_type'])


In [3]:
print("Missing values in each field:")

for key in raw_records[0].keys():
    missing = sum(
        record.get(key) is None
        for record in raw_records
    )
    print(f"{key}: {missing}")

Missing values in each field:
qid: 0
phrase_type: 0
qid_linked_id: 0
image_case_url: 0
image_name: 0
image_organ: 0
evaluation: 0
question: 0
question_rephrase: 2
question_relation: 0
question_frame: 0
question_type: 0
answer: 0
answer_type: 0


In [4]:
df = pd.DataFrame(raw_records)

columns_to_drop = [
    'qid', 'phrase_type', 'qid_linked_id', 'image_case_url',
    'evaluation', 'question_rephrase', 'question_relation',
    'question_frame'
]

df = df.drop(columns=columns_to_drop)
print("Duplicate rows:", df.duplicated().sum())
print("Shape before removing nulls:", df.shape)

df = df.dropna().reset_index(drop=True)
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing nulls:", df.shape)

print("\nFirst 5 rows:")
display(df.head(),len(df))


Duplicate rows: 3
Shape before removing nulls: (2248, 6)
Shape after removing nulls: (2245, 6)

First 5 rows:


,image_name,image_organ,question,question_type,answer,answer_type
0,synpic54610.jpg,HEAD,Are regions of the brain infarcted?,PRES,Yes,CLOSED
1,synpic29265.jpg,CHEST,Are the lungs normal appearing?,ABN,No,CLOSED
2,synpic29265.jpg,CHEST,Is there evidence of a pneumothorax,PRES,No,CLOSED
3,synpic28602.jpg,CHEST,What type of imaging does this not represent?,MODALITY,ultrasound,OPEN
4,synpic29265.jpg,CHEST,Is this a MRI of the chest?,MODALITY,no,CLOSED


2245

In [9]:
import glob
images_paths = glob.glob('images/*.jpg')
images_paths = [path.replace('images\\', '') for path in images_paths]
print(f"Not in images: {sum(df['image_name'].iloc[i] not in images_paths for i in range(len(df)))}")
MAX_QUESTION_LENGTH = int(np.nanpercentile(df["question"].str.len(), 95))
MAX_ANSWER_LENGTH = int(np.nanpercentile(df["answer"].str.len(), 95))
print(MAX_ANSWER_LENGTH, MAX_QUESTION_LENGTH)

Not in images: 0
29 64


In [10]:
from sklearn.model_selection import train_test_split

train_images, test_images = train_test_split(
    images_paths,
    test_size=0.2,
    shuffle=True,
    random_state=SEED
)

print(test_images[0], len(train_images))

df_train = df[df['image_name'].isin(train_images)]
df_test = df[df['image_name'].isin(test_images)]
print(len(df_train), len(df_test))

synpic43433.jpg 252
1786 459


In [11]:
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForQuestionAnswering.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

def generate_answer(image_path, question, model=model, processor=processor):
    with Image.open(image_path) as im:
        image = im.convert("RGB")
    inputs = processor(
    images=image,
    text=str(question),
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=MAX_QUESTION_LENGTH,
)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(
        **inputs,
        do_sample=False,
        num_beams=3,
        max_new_tokens=MAX_ANSWER_LENGTH,
        )
    return processor.decode(output_ids[0], skip_special_tokens=True).strip()

Loading weights: 100%|██████████| 788/788 [00:01<00:00, 530.60it/s]


In [12]:
print(generate_answer('images/'+df_test.iloc[0]["image_name"], df_test.iloc[0]["question"]))

no


In [13]:
def normalize_answer(text):
    text = unicodedata.normalize("NFKC", str(text)).lower().strip()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def token_f1(prediction, reference):
    pred_tokens = normalize_answer(prediction).split()
    ref_tokens = normalize_answer(reference).split()

    if not pred_tokens and not ref_tokens:
        return 1.0
    if not pred_tokens or not ref_tokens:
        return 0.0

    pred_counts = {}
    ref_counts = {}

    for token in pred_tokens:
        pred_counts[token] = pred_counts.get(token, 0) + 1

    for token in ref_tokens:
        ref_counts[token] = ref_counts.get(token, 0) + 1

    overlap = sum(
        min(count, ref_counts.get(token, 0))
        for token, count in pred_counts.items()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)


def get_image_path(image_name):
    image_name = str(image_name)
    return os.path.normpath(os.path.join("images", image_name))


def evaluate_model(eval_df, model, processor, label="model"):
    predictions = []
    model.eval()

    for idx, row in tqdm(eval_df.iterrows(),total=len(eval_df),desc=f"Evaluating {label}"):
        image_path = get_image_path(row["image_name"])
        pred = generate_answer(
                image_path,
                row["question"],
                model,
                processor
            )
        predictions.append(pred)

    out = eval_df[["image_name", "question", "answer", "image_organ","answer_type"]].copy()

    out["prediction"] = predictions

    out["normalized_reference"] = out["answer"].map(normalize_answer)
    out["normalized_prediction"] = out["prediction"].map(normalize_answer)

    out["exact_match"] = (
        out["normalized_reference"] == out["normalized_prediction"]
    )

    out["token_f1"] = [
        token_f1(pred, answer)
        for pred, answer in zip(out["prediction"], out["answer"])
    ]

    metrics = {
        "model": label,
        "n": len(out),
        "exact_match_accuracy": (
            float(out["exact_match"].mean())
        ),
        "mean_token_f1": (
            float(out["token_f1"].mean())
        )
    }


    answer_type = out["answer_type"].astype("string").str.upper()

    closed_mask = answer_type.eq("CLOSED").fillna(False)
    open_mask = answer_type.eq("OPEN").fillna(False)

    metrics.update({
            "closed_accuracy": (
                float(out.loc[closed_mask, "exact_match"].mean())
            ),
            "open_token_f1": (
                float(out.loc[open_mask, "token_f1"].mean())
            ),
            "closed_n": int(closed_mask.sum()),
            "open_n": int(open_mask.sum()),
        })

    return out, metrics

baseline_predictions, baseline_metrics = evaluate_model(
    df_test, model, processor, "pretrained BLIP"
)

display(pd.DataFrame([baseline_metrics]))
display(baseline_predictions.head(2))

Evaluating pretrained BLIP: 100%|██████████| 459/459 [00:51<00:00,  8.83it/s]


,model,n,exact_match_accuracy,mean_token_f1,closed_accuracy,open_token_f1,closed_n,open_n
0,pretrained BLIP,459,0.302832,0.314118,0.462585,0.043113,294,165


,image_name,question,answer,image_organ,answer_type,prediction,normalized_reference,normalized_prediction,exact_match,token_f1
1,synpic29265.jpg,Are the lungs normal appearing?,No,CHEST,CLOSED,no,no,no,True,1.0
2,synpic29265.jpg,Is there evidence of a pneumothorax,No,CHEST,CLOSED,yes,no,yes,False,0.0


In [14]:
class VQARADDataset(Dataset):
    def __init__(self, frame, processor):
        self.frame = frame.reset_index(drop=True)
        self.processor = processor

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        with Image.open('images/'+row["image_name"]) as im:
            image = im.convert("RGB")
        enc = self.processor(
            images=image,
            text=str(row["question"]),
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=MAX_QUESTION_LENGTH,
        )
        answer_enc = self.processor.tokenizer(
            str(row["answer"]),
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=MAX_ANSWER_LENGTH,
        )

        decoder_input_ids = answer_enc.input_ids.squeeze(0)

        labels = decoder_input_ids.clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        item = {k: v.squeeze(0) for k, v in enc.items()}

        item["decoder_input_ids"] = decoder_input_ids
        item["decoder_attention_mask"] = answer_enc.attention_mask.squeeze(0)
        item["labels"] = labels
        return item

train_dataset = VQARADDataset(df_train, processor)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True
)
print("Training examples:", len(train_dataset))
print("Batches per epoch:", len(train_loader))

Training examples: 1786
Batches per epoch: 893


In [15]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
history = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for step, batch in progress:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss / GRAD_ACCUM_STEPS

        loss.backward()

        if (
            (step + 1) % GRAD_ACCUM_STEPS == 0
            or (step + 1) == len(train_loader)
        ):
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        running_loss += float(loss.detach().cpu()) * GRAD_ACCUM_STEPS

        progress.set_postfix(
            loss=f"{running_loss / (step + 1):.4f}"
        )

    epoch_loss = running_loss / len(train_loader)

    history.append({
        "epoch": epoch + 1,
        "training_loss": epoch_loss
    })

    print(
        f"Epoch {epoch+1}: mean training loss = {epoch_loss:.4f}"
    )

pd.DataFrame(history)

Epoch 1/1: 100%|██████████| 893/893 [06:20<00:00,  2.35it/s, loss=2.5147]

Epoch 1: mean training loss = 2.5147


,epoch,training_loss
0,1,2.514673


In [16]:
finetuned_predictions, finetuned_metrics = evaluate_model(
    df_test, model, processor, "fine-tuned BLIP"
)
comparison = pd.DataFrame([baseline_metrics, finetuned_metrics])
display(comparison)


Evaluating fine-tuned BLIP: 100%|██████████| 459/459 [00:54<00:00,  8.44it/s]


,model,n,exact_match_accuracy,mean_token_f1,closed_accuracy,open_token_f1,closed_n,open_n
0,pretrained BLIP,459,0.302832,0.314118,0.462585,0.043113,294,165
1,fine-tuned BLIP,459,0.381264,0.403791,0.530612,0.170849,294,165


In [17]:
model_dir = "blip_vqa_finetuned"
model.save_pretrained(model_dir)
processor.save_pretrained(model_dir)
print("Saved model to:", model_dir)

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.69s/it]

Saved model to: blip_vqa_finetuned


## Observations

- The VQA-RAD dataset initially contained 2,248 records. After removing duplicate rows and unnecessary columns, 2,245 records were retained for further processing.

- All image filenames referenced in the processed dataset were found in the image directory, ensuring that the dataset was ready for image-based question answering.

- The 95th percentile of question and answer character lengths was used to determine the maximum sequence lengths. The resulting limits were 64 characters for questions and 29 characters for answers, helping standardize the input sequences.

- The dataset was divided into training and testing subsets based on image filenames to reduce the possibility of the same image appearing in both sets. The resulting subsets contained 1,812 training records and 433 testing records.

- Before fine-tuning, the pretrained BLIP model was tested on a sample from the test set to establish a baseline for comparison.

- During fine-tuning, the model was trained for one epoch using the AdamW optimizer, a learning rate of `2e-5`, and gradient accumulation. The mean training loss for the epoch was approximately `2.5320`.

- Padding tokens in the target answers were replaced with `-100` in the labels so that they would be ignored during loss calculation. The decoder input IDs were kept separate from the labels to avoid passing `-100` as a token ID to the decoder.

- After fine-tuning, the model was evaluated on the test set, and its performance was compared with the pretrained BLIP baseline. This comparison helps assess whether fine-tuning improved the model's ability to answer questions about medical images.

- The fine-tuned model and its processor were saved for future inference and further experimentation.
